# KIRAVO — Free Kaggle GPU Worker\nEnable T4 x2 + Internet, then run the next cell.\n

In [ ]:
# KIRAVO — Free Kaggle GPU Worker
# Kaggle blocks IPython's os.system() background processes, so this version
# uses Python subprocess + a Flask thread instead of !nohup.

!wget -q https://github.com/Iamkiranofficial/Kiravoo/raw/main/kaggle/kiravo_kaggle_worker.py -O /kaggle/working/kiravo_kaggle_worker.py
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

import sys, subprocess, threading, time, re
from pathlib import Path

sys.path.insert(0, "/kaggle/working")
import kiravo_kaggle_worker as worker

# Start the KIRAVO API inside the current notebook session.
server_thread = threading.Thread(
    target=worker.start_server,
    kwargs={"host": "0.0.0.0", "port": 7860},
    daemon=True,
)
server_thread.start()

# Start Cloudflare Quick Tunnel as a real subprocess.
log_path = "/kaggle/working/kiravo-tunnel.log"
log_file = open(log_path, "w")
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--url",
     "http://127.0.0.1:7860", "--no-autoupdate"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# Wait for the public HTTPS URL.
for _ in range(60):
    text = Path(log_path).read_text(errors="ignore") if Path(log_path).exists() else ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        print("KIRAVO_WORKER_URL =", match.group(0))
        print("KIRAVO worker is running.")
        break
    time.sleep(2)
else:
    print("Tunnel URL not found yet.")
    print(Path(log_path).read_text(errors="ignore")[-4000:])
